# 02 - Advanced RAG Techniques

Advanced RAG techniques: BM25, dense embeddings, hybrid search, query rewriting, cross-encoder reranking, multi-step retrieval, context compression.


In [ ]:
import numpy as np
import re
import math
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
import warnings
warnings.filterwarnings('ignore')

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    return text.split()

documents = [
    "Machine learning is a subset of artificial intelligence that enables computers to learn from data.",
    "Deep learning uses neural networks with many layers to model complex patterns in data.",
    "Natural language processing allows computers to understand and generate human language.",
    "Computer vision enables machines to interpret and understand visual information from images.",
    "Reinforcement learning is a type of machine learning where agents learn by interacting with an environment.",
    "Supervised learning uses labeled data to train models for prediction tasks.",
    "Unsupervised learning finds hidden patterns in data without labeled examples.",
    "Transfer learning allows a model trained on one task to be reused on a related task.",
    "Generative AI can create new content such as text, images, and music.",
    "Large language models are trained on vast amounts of text data to understand and generate language."
]

print('Setup complete!')


## 1. BM25 Implementation


In [ ]:
class BM25:
    def __init__(self, k1=1.5, b=0.75):
        self.k1 = k1
        self.b = b
        self.doc_freqs = []
        self.idf = {}
        self.doc_len = []
        self.avgdl = 0
        self.N = 0
    
    def fit(self, docs):
        tokenized = [preprocess(d) for d in docs]
        self.N = len(docs)
        self.doc_len = [len(t) for t in tokenized]
        self.avgdl = sum(self.doc_len) / self.N
        df = defaultdict(int)
        for tokens in tokenized:
            for word in set(tokens):
                df[word] += 1
        for word, freq in df.items():
            self.idf[word] = math.log((self.N - freq + 0.5) / (freq + 0.5) + 1)
        self.doc_freqs = [Counter(t) for t in tokenized]
        return self
    
    def score(self, query, doc_idx):
        tokens = preprocess(query)
        score = 0
        doc_freq = self.doc_freqs[doc_idx]
        dl = self.doc_len[doc_idx]
        for word in tokens:
            if word in self.idf:
                freq = doc_freq.get(word, 0)
                score += self.idf[word] * (freq * (self.k1 + 1)) / (freq + self.k1 * (1 - self.b + self.b * dl / self.avgdl))
        return score
    
    def get_scores(self, query):
        return [self.score(query, i) for i in range(self.N)]

bm25 = BM25()
bm25.fit(documents)
scores = bm25.get_scores('machine learning')
print(f'BM25 scores: {scores}')


## 2. Dense Embeddings


In [ ]:
class DenseEmbedder:
    def __init__(self, dim=128):
        self.dim = dim
        self.projection = None
    
    def fit(self, docs):
        tokenized = [preprocess(d) for d in docs]
        vocab = set()
        for t in tokenized:
            vocab.update(t)
        self.vocab = sorted(list(vocab))
        self.vocab_index = {w: i for i, w in enumerate(self.vocab)}
        np.random.seed(42)
        self.projection = np.random.randn(len(self.vocab), self.dim) / math.sqrt(len(self.vocab))
        return self
    
    def embed(self, text):
        tokens = preprocess(text)
        vec = np.zeros(len(self.vocab))
        for t in tokens:
            if t in self.vocab_index:
                vec[self.vocab_index[t]] += 1
        dense = vec @ self.projection
        norm = np.linalg.norm(dense)
        return dense / norm if norm > 0 else dense
    
    def embed_batch(self, texts):
        return np.array([self.embed(t) for t in texts])

dense = DenseEmbedder(dim=64)
dense.fit(documents)
embeddings = dense.embed_batch(documents)
print(f'Dense embeddings shape: {embeddings.shape}')


## 3. Hybrid Search


In [ ]:
class HybridSearch:
    def __init__(self, documents, alpha=0.5):
        self.documents = documents
        self.alpha = alpha
        self.bm25 = BM25()
        self.bm25.fit(documents)
        self.dense = DenseEmbedder(dim=64)
        self.dense.fit(documents)
        self.dense_embeddings = self.dense.embed_batch(documents)
    
    def search(self, query, top_k=3):
        sparse_scores = np.array(self.bm25.get_scores(query))
        sparse_scores = (sparse_scores - sparse_scores.min()) / (sparse_scores.max() - sparse_scores.min() + 1e-8)
        query_embed = self.dense.embed(query)
        dense_scores = np.array([np.dot(query_embed, de) for de in self.dense_embeddings])
        hybrid_scores = self.alpha * sparse_scores + (1 - self.alpha) * dense_scores
        top_indices = np.argsort(hybrid_scores)[::-1][:top_k]
        return [(i, hybrid_scores[i], sparse_scores[i], dense_scores[i]) for i in top_indices]

hybrid = HybridSearch(documents, alpha=0.6)
results = hybrid.search('neural networks', top_k=3)
print('Hybrid Search Results:')
for idx, h, s, d in results:
    print(f'  Doc {idx+1}: hybrid={h:.4f}, sparse={s:.4f}, dense={d:.4f}')


## 4. Query Rewriting


In [ ]:
def rewrite_query(query, expansion_terms=None):
    synonyms = {'ml': 'machine learning', 'ai': 'artificial intelligence', 'nn': 'neural network', 'nlp': 'natural language processing', 'cv': 'computer vision'}
    rewritten = query.lower()
    for abbrev, full in synonyms.items():
        rewritten = rewritten.replace(abbrev, full)
    if expansion_terms:
        rewritten += ' ' + ' '.join(expansion_terms)
    return rewritten

queries = ['What is ML?', 'Explain NLP', 'How does CV work?']
for q in queries:
    print(f'{q} -> {rewrite_query(q)}')


## 5. Cross-Encoder Reranking


In [ ]:
class CrossEncoder:
    def score(self, query, document):
        q_tokens = set(preprocess(query))
        d_tokens = set(preprocess(document))
        overlap = len(q_tokens & d_tokens)
        union = len(q_tokens | d_tokens)
        jaccard = overlap / union if union > 0 else 0
        len_ratio = min(len(d_tokens), 20) / 20
        return jaccard * 0.7 + len_ratio * 0.3

def rerank(query, candidates, documents):
    ce = CrossEncoder()
    scored = [(i, ce.score(query, documents[i])) for i, _ in candidates]
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored

query = 'machine learning techniques'
bm25_scores = bm25.get_scores(query)
initial = sorted(enumerate(bm25_scores), key=lambda x: x[1], reverse=True)[:5]
reranked = rerank(query, initial, documents)
print('Before Reranking:')
for i, s in initial:
    print(f'  Doc {i+1}: {s:.4f}')
print('After Reranking:')
for i, s in reranked:
    print(f'  Doc {i+1}: {s:.4f}')


## 6. Multi-Step Retrieval


In [ ]:
class MultiStepRetrieval:
    def __init__(self, documents):
        self.documents = documents
        self.tfidf = None
        self.doc_vectors = None
    
    def fit(self):
        from collections import Counter
        tokenized = [preprocess(d) for d in self.documents]
        all_tokens = set()
        for t in tokenized:
            all_tokens.update(t)
        vocab = sorted(list(all_tokens))
        vocab_index = {w: i for i, w in enumerate(vocab)}
        N = len(self.documents)
        idf = {}
        for word in vocab:
            df = sum(1 for tokens in tokenized if word in tokens)
            idf[word] = math.log(N / (df + 1)) + 1
        vectors = []
        for tokens in tokenized:
            tf = Counter(tokens)
            vec = np.zeros(len(vocab))
            for word, count in tf.items():
                if word in vocab_index:
                    vec[vocab_index[word]] = count * idf[word]
            norm = np.linalg.norm(vec)
            if norm > 0:
                vec = vec / norm
            vectors.append(vec)
        self.doc_vectors = np.array(vectors)
        self.vocab = vocab
        self.vocab_index = vocab_index
        self.idf = idf
        return self
    
    def transform(self, query):
        tokens = preprocess(query)
        tf = Counter(tokens)
        vec = np.zeros(len(self.vocab))
        for word, count in tf.items():
            if word in self.vocab_index:
                vec[self.vocab_index[word]] = count * self.idf[word]
        norm = np.linalg.norm(vec)
        return vec / norm if norm > 0 else vec
    
    def retrieve(self, query, steps=2, docs_per_step=2):
        self.fit()
        all_retrieved = []
        current_query = query
        for step in range(steps):
            query_vec = self.transform(current_query)
            similarities = [(i, np.dot(query_vec, dv)) for i, dv in enumerate(self.doc_vectors)]
            similarities.sort(key=lambda x: x[1], reverse=True)
            step_docs = similarities[:docs_per_step]
            all_retrieved.extend(step_docs)
            new_terms = []
            for idx, _ in step_docs:
                tokens = preprocess(self.documents[idx])
                new_terms.extend(tokens[:3])
            current_query = query + ' ' + ' '.join(new_terms)
        seen = set()
        unique = []
        for idx, score in all_retrieved:
            if idx not in seen:
                seen.add(idx)
                unique.append((idx, score))
        unique.sort(key=lambda x: x[1], reverse=True)
        return unique

msr = MultiStepRetrieval(documents)
results = msr.retrieve('deep learning applications', steps=2, docs_per_step=2)
print('Multi-Step Retrieval Results:')
for idx, score in results:
    print(f'  Doc {idx+1} (score: {score:.4f}): {documents[idx][:60]}...')


## 7. Context Compression


In [ ]:
def compress_context(docs, query, max_tokens=50):
    q_tokens = set(preprocess(query))
    compressed = []
    for doc in docs:
        sentences = doc.split('.')
        scored_sentences = []
        for sent in sentences:
            s_tokens = set(preprocess(sent))
            overlap = len(q_tokens & s_tokens)
            scored_sentences.append((sent.strip(), overlap))
        scored_sentences.sort(key=lambda x: x[1], reverse=True)
        kept = []
        token_count = 0
        for sent, score in scored_sentences:
            if score > 0 and token_count < max_tokens:
                kept.append(sent)
                token_count += len(sent.split())
        compressed.append('. '.join(kept) + '.')
    return compressed

query = 'machine learning'
retrieved = [documents[0], documents[4], documents[5]]
compressed = compress_context(retrieved, query, max_tokens=30)
print('Compressed:')
for i, doc in enumerate(compressed):
    print(f'  {i+1}. {doc}')


## 8. Comparison Visualization


In [ ]:
# Compare methods
query = 'artificial intelligence methods'

# TF-IDF
tokenized = [preprocess(d) for d in documents]
all_tokens = set()
for t in tokenized:
    all_tokens.update(t)
vocab = sorted(list(all_tokens))
vocab_index = {w: i for i, w in enumerate(vocab)}
N = len(documents)
idf = {}
for word in vocab:
    df = sum(1 for tokens in tokenized if word in tokens)
    idf[word] = math.log(N / (df + 1)) + 1
tfidf_vectors = []
for tokens in tokenized:
    tf = Counter(tokens)
    vec = np.zeros(len(vocab))
    for word, count in tf.items():
        if word in vocab_index:
            vec[vocab_index[word]] = count * idf[word]
    norm = np.linalg.norm(vec)
    tfidf_vectors.append(vec / norm if norm > 0 else vec)
tfidf_vectors = np.array(tfidf_vectors)
q_vec = np.zeros(len(vocab))
for word, count in Counter(preprocess(query)).items():
    if word in vocab_index:
        q_vec[vocab_index[word]] = count * idf[word]
norm = np.linalg.norm(q_vec)
q_vec = q_vec / norm if norm > 0 else q_vec
tfidf_scores = [np.dot(q_vec, dv) for dv in tfidf_vectors]

# BM25
bm25_obj = BM25().fit(documents)
bm25_scores = np.array(bm25_obj.get_scores(query))
bm25_scores = (bm25_scores - bm25_scores.min()) / (bm25_scores.max() - bm25_scores.min() + 1e-8)

# Dense
dense_obj = DenseEmbedder(64).fit(documents)
dense_q = dense_obj.embed(query)
dense_scores = [np.dot(dense_q, de) for de in dense_obj.embed_batch(documents)]

x = np.arange(len(documents))
width = 0.25
plt.figure(figsize=(12, 5))
plt.bar(x - width, tfidf_scores, width, label='TF-IDF', alpha=0.8)
plt.bar(x, bm25_scores, width, label='BM25', alpha=0.8)
plt.bar(x + width, dense_scores, width, label='Dense', alpha=0.8)
plt.xlabel('Document Index')
plt.ylabel('Normalized Score')
plt.title('Retrieval Method Comparison')
plt.xticks(x, [f'D{i+1}' for i in range(len(documents))])
plt.legend()
plt.tight_layout()
plt.show()


## 9. Exercises


In [ ]:
# Exercise 1: Use real embeddings (SentenceTransformer)
# Exercise 2: Add query classification
# Exercise 3: Implement feedback loop
# Exercise 4: Test with 100+ documents
print('Exercises listed!')
